In [ ]:
# BETA TUNING

from simulator import run_pytorch_monte_carlo
from util import create_final_matrix
from helpers import (
    calculate_market_rmse,
    safe_scrape_elo,
    country_to_elo,
)
import numpy as np
from constants import TEAM_ID_MAP, ALPHA
import pandas as pd

elo_df = safe_scrape_elo()


# format: [Home, Away, Home_Elo, Away_Elo, Bookie_Odds(H, D, A)]
validation_set = [
    (
        "Switzerland",
        "Colombia",
        country_to_elo(elo_df, "Switzerland"),
        country_to_elo(elo_df, "Colombia"),
        [3.90, 3.19, 2.48],
    ),
    (
        "Argentina",
        "Egypt",
        country_to_elo(elo_df, "Argentina"),
        country_to_elo(elo_df, "Egypt"),
        [1.38, 5.07, 13.00],
    ),
    (
        "USA",
        "Belgium",
        country_to_elo(elo_df, "USA"),
        country_to_elo(elo_df, "Belgium"),
        [2.62, 3.50, 2.98],
    ),
    (
        "Portugal",
        "Spain",
        country_to_elo(elo_df, "Portugal"),
        country_to_elo(elo_df, "Spain"),
        [4.12, 3.67, 2.06],
    ),
    (
        "Mexico",
        "England",
        country_to_elo(elo_df, "Mexico"),
        country_to_elo(elo_df, "England"),
        [3.23, 3.37, 2.60],
    ),
    (
        "Brazil",
        "Norway",
        country_to_elo(elo_df, "Brazil"),
        country_to_elo(elo_df, "Norway"),
        [1.85, 3.86, 5.07],
    ),
]

beta_candidates = np.linspace(start=0.0001, stop=0.0005, num=12)

best_beta = None
lowest_error = 999.0

global_q_matrix = pd.read_csv("./data/global_q_matrix.csv")
global_q_grid = pd.read_csv("./data/global_q_grid.csv")

print("[*] Beginning Beta Grid Search...")

for beta in beta_candidates:
    total_rmse = 0.0
    for match in validation_set:
        home, away, elo_h, elo_a, odds = match
        test_q_grid = create_final_matrix(
            home,
            TEAM_ID_MAP[home],
            away,
            TEAM_ID_MAP[away],
            elo_h,
            elo_a,
            global_q_matrix,
            ALPHA,
            beta,
        )

        p_h, p_d, p_a = run_pytorch_monte_carlo(test_q_grid, num_simulations=5000)
        model_probs = [p_h, p_d, p_a]
        error = calculate_market_rmse(model_probs, odds)
        total_rmse += error

    avg_rmse = total_rmse / len(validation_set)
    print(f"Beta: {beta:.6f} | Average Market RMSE: {avg_rmse:.4f}")

    if avg_rmse < lowest_error:
        best_beta = beta
        lowest_error = avg_rmse
        print(f"New best beta found: {best_beta}!")
print(f"\n[+] Optimal Beta Found: {best_beta:.6f} (Error: {lowest_error:.4f})")

In [ ]:
# SIMULTANEOUS ALPHA AND BETA TUNING
# this must be run on my GPU since it'll take 38 min on my laptop


from simulator import run_pytorch_monte_carlo
from util import create_final_matrix
from helpers import (
    calculate_market_rmse,
    safe_scrape_elo,
    country_to_elo,
)
import numpy as np
from constants import TEAM_ID_MAP
import pandas as pd
from tqdm import tqdm

elo_df = safe_scrape_elo()


# format: [Home, Away, Home_Elo, Away_Elo, Bookie_Odds(H, D, A)]
validation_set = [
    (
        "Switzerland",
        "Colombia",
        country_to_elo(elo_df, "Switzerland"),
        country_to_elo(elo_df, "Colombia"),
        [3.90, 3.19, 2.48],
    ),
    (
        "Argentina",
        "Egypt",
        country_to_elo(elo_df, "Argentina"),
        country_to_elo(elo_df, "Egypt"),
        [1.38, 5.07, 13.00],
    ),
    (
        "USA",
        "Belgium",
        country_to_elo(elo_df, "USA"),
        country_to_elo(elo_df, "Belgium"),
        [2.62, 3.50, 2.98],
    ),
    (
        "Portugal",
        "Spain",
        country_to_elo(elo_df, "Portugal"),
        country_to_elo(elo_df, "Spain"),
        [4.12, 3.67, 2.06],
    ),
    (
        "Mexico",
        "England",
        country_to_elo(elo_df, "Mexico"),
        country_to_elo(elo_df, "England"),
        [3.23, 3.37, 2.60],
    ),
    (
        "Brazil",
        "Norway",
        country_to_elo(elo_df, "Brazil"),
        country_to_elo(elo_df, "Norway"),
        [1.85, 3.86, 5.07],
    ),
]
alpha_candidates = np.linspace(start=0.01, stop=0.6, num=10)
beta_candidates = np.linspace(start=0.0001, stop=0.0006, num=10)

total_iterations = len(alpha_candidates) * len(beta_candidates)

best_alpha = None
best_beta = None
lowest_error = 999.0

global_q_matrix = pd.read_csv("./data/global_q_matrix.csv")
global_q_grid = pd.read_csv("./data/global_q_grid.csv")

print(
    f"[*] Beginning 2D Grid Search ({len(alpha_candidates) * len(beta_candidates)} combos)..."
)
with tqdm(
    total=total_iterations, desc="Optimising Hyperparameters", unit="grid"
) as pbar:
    for alpha in alpha_candidates:
        for beta in beta_candidates:
            total_rmse = 0.0

            for match in validation_set:
                home, away, elo_h, elo_a, odds = match

                # Pass BOTH dynamic hyperparameters
                test_q_grid = create_final_matrix(
                    home,
                    TEAM_ID_MAP[home],
                    away,
                    TEAM_ID_MAP[away],
                    elo_h,
                    elo_a,
                    global_q_matrix,
                    alpha,
                    beta,
                )

                p_h, p_d, p_a = run_pytorch_monte_carlo(
                    test_q_grid, num_simulations=3000
                )

                model_probs = [p_h, p_d, p_a]

                error = calculate_market_rmse(model_probs, odds)
                total_rmse += error

            avg_rmse = total_rmse / len(validation_set)

            if avg_rmse < lowest_error:
                tqdm.write(
                    f"[!] New best found! Alpha: {alpha:.4f} | Beta: {beta:.6f} | RMSE: {avg_rmse:.4f}"
                )
                lowest_error = avg_rmse
                best_alpha = alpha
                best_beta = beta
            pbar.update(1)

print("\n=========================================")
print("          OPTIMAL HYPERPARAMETERS        ")
print("=========================================")
print(f"Alpha: {best_alpha:.4f}")
print(f"Beta:  {best_beta:.6f}")
print(f"Final Market RMSE: {lowest_error:.4f}")
print("=========================================\n")

[*] Beginning 2D Grid Search (100 combos)...


Optimising Hyperparameters:   0%|          | 0/100 [00:00<?, ?grid/s]

[*] Found 4 matches for Switzerland. Parsing...
[+] Successfully built dataframe for Switzerland (5067 events).
[*] Found 4 matches for Colombia. Parsing...
[+] Successfully built dataframe for Colombia (4918 events).

[*] Prepping matrix for 3000 parallel simulations...
[*] Compute Device: CUDA
[*] Firing PyTorch Engine...

      FINAL MATCH FORECAST        
Hardware Time: 1.933 seconds
Home Win:  40.60%
Draw:      18.93%
Away Win:  40.47%
----------------------------------
Expected Score: Home 2.11 - 2.12 Away

[*] Found 4 matches for Argentina. Parsing...
[+] Successfully built dataframe for Argentina (6002 events).
[*] Found 4 matches for Egypt. Parsing...
[+] Successfully built dataframe for Egypt (5715 events).

[*] Prepping matrix for 3000 parallel simulations...
[*] Compute Device: CUDA
[*] Firing PyTorch Engine...

      FINAL MATCH FORECAST        
Hardware Time: 2.033 seconds
Home Win:  74.53%
Draw:      12.60%
Away Win:  12.87%
----------------------------------
Expected Sc

Optimising Hyperparameters:   1%|          | 1/100 [00:14<23:36, 14.31s/grid]


      FINAL MATCH FORECAST        
Hardware Time: 1.979 seconds
Home Win:  37.70%
Draw:      19.77%
Away Win:  42.53%
----------------------------------
Expected Score: Home 2.20 - 2.31 Away

[!] New best found! Alpha: 0.0100 | Beta: 0.000100 | RMSE: 0.1259
[*] Found 4 matches for Switzerland. Parsing...
[+] Successfully built dataframe for Switzerland (5067 events).
[*] Found 4 matches for Colombia. Parsing...
[+] Successfully built dataframe for Colombia (4918 events).

[*] Prepping matrix for 3000 parallel simulations...
[*] Compute Device: CUDA
[*] Firing PyTorch Engine...

      FINAL MATCH FORECAST        
Hardware Time: 1.958 seconds
Home Win:  39.67%
Draw:      20.37%
Away Win:  39.97%
----------------------------------
Expected Score: Home 2.07 - 2.07 Away

[*] Found 4 matches for Argentina. Parsing...
[+] Successfully built dataframe for Argentina (6002 events).
[*] Found 4 matches for Egypt. Parsing...
[+] Successfully built dataframe for Egypt (5715 events).

[*] Prepping 

...


      FINAL MATCH FORECAST        
Hardware Time: 1.965 seconds
Home Win:  47.77%
Draw:      21.40%
Away Win:  30.83%
----------------------------------
Expected Score: Home 2.07 - 1.69 Away

[*] Found 4 matches for Switzerland. Parsing...
[+] Successfully built dataframe for Switzerland (5067 events).
[*] Found 4 matches for Colombia. Parsing...
[+] Successfully built dataframe for Colombia (4918 events).

[*] Prepping matrix for 3000 parallel simulations...
[*] Compute Device: CUDA
[*] Firing PyTorch Engine...

      FINAL MATCH FORECAST        
Hardware Time: 1.967 seconds
Home Win:  41.27%
Draw:      22.03%
Away Win:  36.70%
----------------------------------
Expected Score: Home 1.86 - 1.77 Away

[*] Found 4 matches for Argentina. Parsing...
[+] Successfully built dataframe for Argentina (6002 events).
[*] Found 4 matches for Egypt. Parsing...
[+] Successfully built dataframe for Egypt (5715 events).

[*] Prepping matrix for 3000 parallel simulations...
[*] Compute Device: CUDA
[

Optimising Hyperparameters:  99%|█████████▉| 99/100 [23:50<00:14, 14.16s/grid]


      FINAL MATCH FORECAST        
Hardware Time: 1.977 seconds
Home Win:  50.70%
Draw:      20.13%
Away Win:  29.17%
----------------------------------
Expected Score: Home 2.11 - 1.62 Away

[*] Found 4 matches for Switzerland. Parsing...
[+] Successfully built dataframe for Switzerland (5067 events).
[*] Found 4 matches for Colombia. Parsing...
[+] Successfully built dataframe for Colombia (4918 events).

[*] Prepping matrix for 3000 parallel simulations...
[*] Compute Device: CUDA
[*] Firing PyTorch Engine...

      FINAL MATCH FORECAST        
Hardware Time: 2.020 seconds
Home Win:  41.27%
Draw:      21.67%
Away Win:  37.07%
----------------------------------
Expected Score: Home 1.85 - 1.75 Away

[*] Found 4 matches for Argentina. Parsing...
[+] Successfully built dataframe for Argentina (6002 events).
[*] Found 4 matches for Egypt. Parsing...
[+] Successfully built dataframe for Egypt (5715 events).

[*] Prepping matrix for 3000 parallel simulations...
[*] Compute Device: CUDA
[

Optimising Hyperparameters: 100%|██████████| 100/100 [24:05<00:00, 14.45s/grid]


      FINAL MATCH FORECAST        
Hardware Time: 1.962 seconds
Home Win:  48.00%
Draw:      20.73%
Away Win:  31.27%
----------------------------------
Expected Score: Home 2.06 - 1.60 Away


          OPTIMAL HYPERPARAMETERS        
Alpha: 0.3000
Beta:  0.000322
Final Market RMSE: 0.0774

